# Práctica de Clase 2 — *Cuando el camino más corto no es el mejor*

Prototipo de búsqueda de rutas de evacuación en una zona inundada. El agente debe llegar desde un punto inicial hasta un refugio eligiendo la ruta **más conveniente** (menor costo combinado distancia + riesgo), no necesariamente la más corta.

| Integrante | Sección del notebook |
|---|---|
| Jesús | 2. Modelado del terreno y función de costo |
| Martín | 3. Algoritmos de búsqueda e instrumentación |
| Izquierdo | 4. Visualización |
| Lechuga | 5. Escenarios de prueba |
| Ramón | 1. Metodología · 6. Verificación y análisis comparativo · 7. Declaración de IA y reproducibilidad |

### Índice
1. Introducción, metodología y respuesta a la pregunta eje
2. Modelado del terreno y función de costo
3. Algoritmos de búsqueda e instrumentación
4. Visualización
5. Escenarios de prueba
6. Verificación independiente y análisis comparativo
7. Declaración de uso de IA y reproducibilidad

> **Ejecución:** `Kernel → Restart & Run All`. Las celdas dependen de las anteriores; no ejecutar fuera de orden.

## 1. Introducción y metodología

### 1.1 Planteamiento del problema
Tras una inundación, la ruta geométricamente más corta entre una zona afectada y un refugio puede cruzar calles con agua profunda, corriente o escombros. Minimizar sólo la distancia (euclidiana o en pasos) trata todas las celdas como si fueran igual de seguras, por lo que puede **encontrar una ruta** que no es una **ruta conveniente**. El problema se modela entonces como un **camino de costo mínimo en un grafo ponderado**, donde el peso de cada movimiento incorpora el riesgo del terreno.

### 1.2 Metodología
1. **Modelado discreto del terreno.** Cuadrícula `rows × cols`; cada celda tiene un nivel `-1` (bloqueo total), `0` (seco) o `1–3` (anegación creciente). Cada celda es un nodo; las aristas conectan celdas vecinas transitables.
2. **Función de costo.** $C(u,v)=d(u,v)\,(1+\alpha R(v)^{\beta})$ y $C=\infty$ si $v$ está bloqueada. Es siempre $\ge d(u,v)>0$, por lo que no hay pesos negativos (requisito de Dijkstra) y la distancia pura es una cota inferior del costo real (base de una heurística admisible para A\*).
3. **Estrategias de búsqueda.** Se comparan una búsqueda no informada (Dijkstra / costo uniforme) y una informada (A\* con heurística admisible). Ambas deben devolver el mismo costo óptimo; la diferencia esperada está en el **trabajo** realizado (nodos expandidos, tiempo).
4. **Instrumentación.** Cada ejecución registra: si encontró ruta, la ruta, su costo acumulado, su longitud en pasos, nodos expandidos y tiempo de cómputo.
5. **Verificación independiente (sección 6).** Cada ruta reportada se valida con código ajeno a los algoritmos: extremos correctos, adyacencia, ausencia de celdas bloqueadas, ausencia de cortes de esquina, y **recálculo del costo** para compararlo con el que reporta el algoritmo. Si no se encuentra ruta, se comprueba con un BFS independiente que efectivamente no existe.
6. **Escenarios.** Se ejecutan los escenarios obligatorios (ruta base, bloqueo dinámico, refugio inaccesible, aumento de riesgo, corto vs. conveniente) y un escenario adicional diseñado para detectar un error de la propia implementación.
7. **Comparación.** La tabla de resultados se **genera automáticamente** a partir de las ejecuciones reales del notebook; no se escriben valores a mano.

### 1.3 Pregunta eje
> *Si dos rutas llegan al mismo refugio, pero una es más corta y la otra atraviesa zonas de mayor riesgo, ¿cómo determina el sistema cuál es la mejor?*

**Respuesta.** El sistema no compara longitudes, compara el **costo acumulado** de cada ruta:

$$J(P)=\sum_{(u,v)\in P} d(u,v)\,\bigl(1+\alpha R(v)^{\beta}\bigr)$$

y elige la ruta con menor $J(P)$. Como Dijkstra y A\* minimizan exactamente $J$, la decisión no es una regla añadida al final, sino una consecuencia directa de cómo se ponderan las aristas.

**Ejemplo analítico** (cálculo directo con la fórmula, $\alpha=2$, $\beta=1.5$, movimientos ortogonales; *no es una medición*):

| Ruta | Composición | Pasos | $J(P)$ |
|---|---|---|---|
| A (corta) | 4 celdas nivel 3 + 2 secas | 6 | $4(11.392)+2 = 47.57$ |
| B (rodeo seco) | 12 celdas secas | 12 | $12.00$ |
| C (rodeo leve) | 8 celdas nivel 1 | 8 | $8(3)=24.00$ |

El sistema elige **B** aunque sea el doble de larga. En general, un rodeo de $k$ pasos secos es preferible a cruzar $n$ celdas de nivel $R$ mientras $k < n\,(1+\alpha R^{\beta})$.

- $\alpha$ controla la aversión al riesgo: con $\alpha=0$ el sistema se reduce a *camino más corto* y elegiría A; al aumentar $\alpha$ se favorecen los desvíos.
- $\beta>1$ castiga el agua profunda más que proporcionalmente: 1 celda de nivel 3 (11.39) cuesta más que 3 celdas de nivel 1 (9.00).
- Si toda conexión con el refugio pasa por celdas `-1`, $J=\infty$ y el sistema reporta **“sin ruta”** en vez de devolver un camino inválido.

La demostración **empírica** de esta respuesta está en el Escenario 5 (sección 5) y en la tabla de la sección 6, donde la ruta elegida se compara contra la ruta más corta en pasos.

# Cuando el camino más corto no es el mejor — Modelado del terreno y función de costo

**Problema:** evacuar una zona inundada hacia un refugio. La ruta más corta puede cruzar zonas de alto riesgo.

**Pregunta eje:** *Si dos rutas llegan al mismo refugio, pero una es más corta y la otra atraviesa zonas de mayor riesgo, ¿cómo determina el sistema cuál es la mejor?*
Comparando el costo acumulado $J(P)=\sum C(u,v)$, no la longitud.

## Función de costo
$$C(u,v)=\begin{cases} d(u,v)\,\bigl(1+\alpha\,R(v)^{\beta}\bigr) & v \text{ transitable}\\ \infty & R(v)=-1\end{cases}$$

- $d(u,v)$: 1 (ortogonal) o $\sqrt2$ (diagonal).
- $R(v)$: nivel de riesgo de la celda destino (0 seco, 1–3 anegación, -1 bloqueo total).
- $\alpha$: aversión al riesgo. $\beta$: exponente no lineal.

| Nivel | Significado | Factor (α=2, β=1.5) |
|---|---|---|
| 0 | Seco | 1.00 |
| 1 | Leve | 3.00 |
| 2 | Moderado | 6.66 |
| 3 | Severo | 11.39 |
| -1 | Bloqueo | ∞ |


In [ ]:
from __future__ import annotations

import math

import numpy as np

Node = tuple[int, int]


class GridEnvironment:
    """Cuadrícula de riesgo de inundación con función de costo de transición.

    Niveles: 0 seco, 1-3 anegación creciente, -1 bloqueo total.
    Costo: C(u, v) = d(u, v) * (1 + alpha * R(v) ** beta); inf si v está bloqueada.
    """

    BLOCKED = -1
    MAX_RISK = 3
    _ORTHO = ((-1, 0), (1, 0), (0, -1), (0, 1))
    _DIAG = ((-1, -1), (-1, 1), (1, -1), (1, 1))

    def __init__(self, rows: int, cols: int, alpha: float = 2.0,
                 beta: float = 1.5, allow_diagonal: bool = False) -> None:
        if rows <= 0 or cols <= 0:
            raise ValueError("rows y cols deben ser positivos")
        if alpha < 0 or beta < 1:
            raise ValueError("se requiere alpha >= 0 y beta >= 1")
        self.rows = rows
        self.cols = cols
        self.alpha = alpha
        self.beta = beta
        self.allow_diagonal = allow_diagonal
        self.grid: np.ndarray = np.zeros((rows, cols), dtype=np.int8)

    # ---- validación ----
    def in_bounds(self, r: int, c: int) -> bool:
        """True si (r, c) está dentro de la cuadrícula."""
        return 0 <= r < self.rows and 0 <= c < self.cols

    def _check_level(self, risk_level: int) -> None:
        if risk_level != self.BLOCKED and not 0 <= risk_level <= self.MAX_RISK:
            raise ValueError(f"risk_level inválido: {risk_level} (use -1 o 0..3)")

    # ---- modificación del terreno ----
    def set_cell(self, r: int, c: int, risk_level: int) -> None:
        """Asigna el nivel de riesgo de una celda."""
        if not self.in_bounds(r, c):
            raise IndexError(f"celda fuera de límites: ({r}, {c})")
        self._check_level(risk_level)
        self.grid[r, c] = risk_level

    def set_region_risk(self, r_start: int, r_end: int, c_start: int,
                        c_end: int, risk_level: int) -> None:
        """Asigna un nivel a la región rectangular [r_start, r_end] x [c_start, c_end] (inclusive)."""
        if r_start > r_end or c_start > c_end:
            raise ValueError("rango de región invertido")
        if not (self.in_bounds(r_start, c_start) and self.in_bounds(r_end, c_end)):
            raise IndexError("región fuera de límites")
        self._check_level(risk_level)
        self.grid[r_start:r_end + 1, c_start:c_end + 1] = risk_level

    def load_matrix(self, matrix: np.ndarray | list[list[int]]) -> None:
        """Carga una matriz 2D completa validando dimensiones y niveles (-1 o 0..3)."""
        arr = np.asarray(matrix, dtype=np.int8)
        if arr.shape != (self.rows, self.cols):
            raise ValueError(f"dimensiones inválidas: se esperaba {(self.rows, self.cols)}, se recibió {arr.shape}")
        if not np.isin(arr, (-1, 0, 1, 2, 3)).all():
            raise ValueError("la matriz contiene niveles fuera de {-1, 0, 1, 2, 3}")
        self.grid[:, :] = arr

    def reset(self) -> None:
        """Limpia el terreno a ceros (todo seco)."""
        self.grid.fill(0)

    # ---- consultas ----
    def is_blocked(self, r: int, c: int) -> bool:
        """True si la celda es intransitable (bloqueada o fuera de límites)."""
        return (not self.in_bounds(r, c)) or self.grid[r, c] == self.BLOCKED

    def cost(self, u: Node, v: Node) -> float:
        """Costo de transición u -> v (v adyacente). inf si v está bloqueada."""
        if self.is_blocked(*v):
            return math.inf
        dr, dc = abs(u[0] - v[0]), abs(u[1] - v[1])
        if max(dr, dc) != 1:
            raise ValueError(f"{u} y {v} no son adyacentes")
        d = math.sqrt(2.0) if dr + dc == 2 else 1.0
        risk = float(self.grid[v])
        return d * (1.0 + self.alpha * risk ** self.beta)

    def get_neighbors(self, node: Node) -> list[tuple[Node, float]]:
        """Vecinos transitables de `node` como ((r, c), costo). Nunca incluye bloqueados."""
        r, c = node
        if not self.in_bounds(r, c):
            raise IndexError(f"nodo fuera de límites: {node}")
        result: list[tuple[Node, float]] = []
        for dr, dc in self._ORTHO:
            nr, nc = r + dr, c + dc
            if not self.is_blocked(nr, nc):
                result.append(((nr, nc), self.cost(node, (nr, nc))))
        if self.allow_diagonal:
            for dr, dc in self._DIAG:
                nr, nc = r + dr, c + dc
                if self.is_blocked(nr, nc):
                    continue
                # Anti corner-cutting: impide diagonal cuando las dos ortogonales intermedias están bloqueadas
                if self.is_blocked(r + dr, c) and self.is_blocked(r, c + dc):
                    continue
                result.append(((nr, nc), self.cost(node, (nr, nc))))
        return result


In [ ]:
# Smoke test: mapa 5x5 con un bloqueo
env = GridEnvironment(5, 5)
env.set_cell(2, 2, -1)
assert env.is_blocked(2, 2)
nb = dict(env.get_neighbors((2, 1)))
assert (2, 2) not in nb and set(nb) == {(1, 1), (3, 1), (2, 0)}, nb
print("Smoke test OK:", nb)

# Costos ponderados (alpha=2, beta=1.5)
env.reset()
env.set_cell(1, 0, 1)
env.set_cell(0, 1, 3)
assert math.isclose(env.cost((0, 0), (1, 0)), 1 + 2 * 1 ** 1.5)      # 3.0
assert math.isclose(env.cost((0, 0), (0, 1)), 1 + 2 * 3 ** 1.5)      # ~11.39
assert env.cost((0, 0), (1, 0)) < env.cost((0, 0), (0, 1))           # más riesgo, más costo
env.set_cell(0, 1, -1)
assert env.cost((0, 0), (0, 1)) == math.inf
assert (0, 1) not in dict(env.get_neighbors((0, 0)))

# Región, carga matricial y reset
env.set_region_risk(0, 1, 0, 1, 2)
assert (env.grid[0:2, 0:2] == 2).all()
env.load_matrix(np.zeros((5, 5), dtype=np.int8))
assert (env.grid == 0).all()
env.reset()
assert (env.grid == 0).all()

# Diagonales: costo sqrt(2) y anti corner-cutting
d = GridEnvironment(3, 3, allow_diagonal=True)
assert math.isclose(dict(d.get_neighbors((1, 1)))[(0, 0)], math.sqrt(2))
d.set_cell(0, 1, -1)
d.set_cell(1, 0, -1)
assert (0, 0) not in dict(d.get_neighbors((1, 1)))   # ambas bloqueadas -> impide corte de esquina
d.set_cell(1, 0, 0)
assert (0, 0) in dict(d.get_neighbors((1, 1)))       # solo una bloqueada -> permitido

# Validaciones
for bad in (lambda: env.set_cell(9, 9, 0), lambda: env.set_cell(0, 0, 7)):
    try:
        bad()
        raise AssertionError("debió fallar")
    except (IndexError, ValueError):
        pass
print("Todas las pruebas pasaron")


## 3. Algoritmos de búsqueda e instrumentación *(Martín)*

> **Contrato de interfaz** acordado para que las secciones 5 y 6 funcionen sin cambios. Cada algoritmo debe tener la firma `algoritmo(env, start, goal) -> dict` y devolver:
>
> | Clave | Tipo | Significado |
> |---|---|---|
> | `found` | `bool` | `True` si se llegó al refugio |
> | `path` | `list[Node]` | ruta de `start` a `goal` inclusive; `[]` si no hay ruta |
> | `cost` | `float` | costo acumulado $J(P)$; `math.inf` si no hay ruta |
> | `nodes_expanded` | `int` | nodos sacados de la frontera y expandidos |
> | `time_ms` | `float` | tiempo de cómputo medido con `time.perf_counter()` |
>
> Si el refugio es inalcanzable **no se lanza excepción**: se devuelve `found=False`. Con `allow_diagonal=True` la heurística de A\* debe ser la distancia **octil** (Manhattan deja de ser admisible).

## 4. Visualización *(Izquierdo)*

> Mapa de calor del riesgo, celdas bloqueadas, inicio, refugio y rutas sobrepuestas; gráficas de barras de nodos expandidos y costo por algoritmo (pueden consumir la lista `resultados` de la sección 6).

## 5. Escenarios de prueba *(Lechuga)*

> Escenarios 1–5 obligatorios + escenario 6 caza-errores. Para alimentar la sección 6, cada ejecución se registra con:
>
> ```python
> registrar("E1 Base", "A*", env, start, goal, a_star(env, start, goal))
> ```

## 6. Verificación independiente y análisis comparativo *(Ramón)*

Las funciones siguientes **no reutilizan la lógica de búsqueda**: validan cada resultado a partir de la definición del problema. Así una ruta sólo se acepta si:

1. empieza en el inicio y termina en el refugio;
2. cada paso es entre celdas adyacentes (ortogonales, o diagonales sólo si `allow_diagonal=True`);
3. no pisa celdas bloqueadas ni fuera de la cuadrícula;
4. no repite celdas (una ruta óptima con costos positivos nunca tiene ciclos);
5. no hace cortes de esquina (diagonal junto a una celda bloqueada);
6. el costo reportado coincide con el costo **recalculado** paso a paso con `env.cost`.

Si un algoritmo reporta `found=False`, se comprueba con un BFS independiente que **realmente** no existe camino. La tabla comparativa se construye sólo con los registros de ejecuciones reales almacenados en `resultados`.

In [ ]:
from collections import deque


def existe_camino(env: GridEnvironment, start: Node, goal: Node) -> bool:
    """BFS independiente (ignora costos): True si goal es alcanzable desde start."""
    if env.is_blocked(*start) or env.is_blocked(*goal):
        return False
    vistos = {start}
    cola = deque([start])
    while cola:
        u = cola.popleft()
        if u == goal:
            return True
        for v, _ in env.get_neighbors(u):
            if v not in vistos:
                vistos.add(v)
                cola.append(v)
    return False


def costo_recalculado(env: GridEnvironment, path: list[Node]) -> float:
    """Suma env.cost(u, v) a lo largo de la ruta (J(P) recalculado)."""
    return sum(env.cost(u, v) for u, v in zip(path, path[1:]))


def verificar_ruta(env: GridEnvironment, start: Node, goal: Node,
                   res: dict, tol: float = 1e-6) -> list[str]:
    """Devuelve la lista de violaciones encontradas; lista vacía = resultado válido."""
    errores: list[str] = []
    path = list(res.get("path") or [])

    if not res.get("found"):
        if path:
            errores.append("found=False pero se devolvió una ruta")
        if res.get("cost", math.inf) != math.inf:
            errores.append("found=False pero el costo no es inf")
        if existe_camino(env, start, goal):
            errores.append("reportó 'sin ruta' pero el BFS independiente SÍ encontró camino")
        return errores

    if not path:
        return ["found=True pero la ruta está vacía"]
    if path[0] != start:
        errores.append(f"la ruta empieza en {path[0]}, no en {start}")
    if path[-1] != goal:
        errores.append(f"la ruta termina en {path[-1]}, no en {goal}")
    if len(set(path)) != len(path):
        errores.append("la ruta repite celdas (ciclo)")

    for celda in path:
        if env.is_blocked(*celda):
            errores.append(f"la ruta pisa una celda bloqueada o fuera de límites: {celda}")

    for (r1, c1), (r2, c2) in zip(path, path[1:]):
        dr, dc = r2 - r1, c2 - c1
        if max(abs(dr), abs(dc)) != 1:
            errores.append(f"salto no adyacente {(r1, c1)} -> {(r2, c2)}")
        elif dr != 0 and dc != 0:
            if not env.allow_diagonal:
                errores.append(f"diagonal no permitida {(r1, c1)} -> {(r2, c2)}")
            elif env.is_blocked(r1 + dr, c1) or env.is_blocked(r1, c1 + dc):
                errores.append(f"corte de esquina junto a celda bloqueada {(r1, c1)} -> {(r2, c2)}")

    if not errores:
        real = costo_recalculado(env, path)
        if not math.isclose(real, res.get("cost", math.nan), rel_tol=tol, abs_tol=tol):
            errores.append(f"costo reportado {res.get('cost')} != recalculado {real:.4f}")
    return errores


resultados: list[dict] = []


def registrar(escenario: str, algoritmo: str, env: GridEnvironment,
              start: Node, goal: Node, res: dict) -> dict:
    """Verifica un resultado real y lo agrega a `resultados` para la tabla comparativa."""
    errores = verificar_ruta(env, start, goal, res)
    fila = {
        "escenario": escenario,
        "algoritmo": algoritmo,
        "found": bool(res.get("found")),
        "cost": res.get("cost", math.inf),
        "steps": max(len(res.get("path") or []) - 1, 0),
        "nodes_expanded": res.get("nodes_expanded"),
        "time_ms": res.get("time_ms"),
        "errores": errores,
    }
    resultados.append(fila)
    estado = "OK" if not errores else "FALLA: " + "; ".join(errores)
    print(f"[{escenario} | {algoritmo}] verificación: {estado}")
    return fila


def tabla_comparativa(filas: list[dict] | None = None) -> str:
    """Construye una tabla Markdown SOLO con ejecuciones registradas (nada estimado)."""
    filas = resultados if filas is None else filas
    if not filas:
        return "_Aún no hay ejecuciones registradas: ejecute primero la sección 5._"

    def fmt(x, nd=2):
        if x is None:
            return "—"
        if isinstance(x, float):
            return "∞" if math.isinf(x) else f"{x:.{nd}f}"
        return str(x)

    lineas = [
        "| Escenario | Algoritmo | ¿Ruta? | Costo J(P) | Pasos | Nodos expandidos | Tiempo (ms) | Verificación |",
        "|---|---|---|---|---|---|---|---|",
    ]
    for f in filas:
        lineas.append(
            f"| {f['escenario']} | {f['algoritmo']} | {'sí' if f['found'] else 'no'} | "
            f"{fmt(f['cost'])} | {f['steps'] if f['found'] else '—'} | {fmt(f['nodes_expanded'])} | "
            f"{fmt(f['time_ms'], 3)} | {'OK' if not f['errores'] else 'FALLA'} |"
        )
    return "\n".join(lineas)


# Autoprueba del verificador con rutas construidas a mano (no son resultados de algoritmos)
_e = GridEnvironment(3, 3)
_e.set_cell(1, 1, -1)
_ok = {"found": True, "path": [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)], "cost": 4.0}
assert verificar_ruta(_e, (0, 0), (2, 2), _ok) == []
assert verificar_ruta(_e, (0, 0), (2, 2), {**_ok, "cost": 3.0})                       # costo mal reportado
assert verificar_ruta(_e, (0, 0), (2, 2), {**_ok, "path": [(0, 0), (1, 1), (2, 2)]})  # cruza bloqueo
assert verificar_ruta(_e, (0, 0), (2, 2), {"found": False, "path": [], "cost": math.inf})  # sí había camino
_e.set_region_risk(0, 2, 1, 1, -1)
assert verificar_ruta(_e, (0, 0), (2, 2), {"found": False, "path": [], "cost": math.inf}) == []
print("Verificador independiente OK")


In [ ]:
# Tabla comparativa generada a partir de las ejecuciones reales registradas en la sección 5
try:
    from IPython.display import Markdown, display
    display(Markdown(tabla_comparativa()))
except ImportError:
    print(tabla_comparativa())

fallas = [f for f in resultados if f["errores"]]
print(f"Ejecuciones registradas: {len(resultados)} | con fallas de verificación: {len(fallas)}")

# Comprobación cruzada: ambos algoritmos óptimos deben coincidir en costo dentro de cada escenario
por_escenario: dict[str, list[dict]] = {}
for f in resultados:
    por_escenario.setdefault(f["escenario"], []).append(f)
for esc, filas in por_escenario.items():
    costos = {f["algoritmo"]: f["cost"] for f in filas}
    vals = list(costos.values())
    iguales = all(
        (math.isinf(a) and math.isinf(b)) or math.isclose(a, b, rel_tol=1e-9)
        for a, b in zip(vals, vals[1:])
    )
    print(f"{esc}: costos {costos} -> {'coinciden' if iguales else 'NO COINCIDEN (revisar heurística)'}")


### 6.1 Análisis comparativo

> ⚠️ **Completar sólo después de `Restart & Run All`**, copiando los valores de la tabla anterior. No escribir cifras que no aparezcan en la salida.

**Calidad de la ruta.** Dijkstra y A\* minimizan la misma función $J(P)$; con una heurística admisible ambos deben obtener el **mismo costo** en cada escenario (lo comprueba automáticamente la celda anterior). En los resultados: ⟨indicar si coincidieron en todos los escenarios; si no, en cuál y por qué⟩.

**Trabajo realizado.** Dijkstra expande en orden de costo acumulado $g(n)$, sin información sobre la ubicación del refugio, por lo que crece en todas direcciones. A\* ordena por $f(n)=g(n)+h(n)$ y prioriza nodos orientados al refugio. En los resultados: ⟨nodos expandidos de cada algoritmo por escenario y diferencia relativa⟩.

**Por qué la ventaja de A\* se reduce con riesgo alto.** La heurística sólo estima la distancia ($h \le$ costo real porque el factor de riesgo es $\ge 1$). Cuando la ruta óptima es un desvío caro (escenarios 4 y 5), el costo real supera mucho a $h$, la heurística queda “optimista” y A\* termina explorando casi como Dijkstra. ⟨confirmar o refutar con los datos de E4/E5⟩.

**Refugio inaccesible (E3).** Ambos algoritmos deben agotar la componente conexa alcanzable antes de concluir “sin ruta”, de modo que expanden aproximadamente la misma cantidad de nodos. ⟨valores observados y confirmación del BFS independiente⟩.

**Corto vs. conveniente (E5).** ⟨longitud en pasos y costo de la ruta elegida frente a la ruta más corta en pasos⟩. Esto demuestra empíricamente la respuesta de la sección 1.3: el sistema eligió la ruta de menor $J(P)$, no la de menos pasos.

**Tiempo de cómputo.** Las mediciones en milisegundos dependen de la máquina y varían entre ejecuciones; se usan como referencia secundaria. La métrica principal de trabajo es `nodes_expanded`, que es determinista.

**Escenario 6 (caza-errores).** ⟨qué error de la propia implementación buscaba detectar, si lo detectó y qué se corrigió⟩.

## 7. Declaración de uso de IA y reproducibilidad *(Ramón)*

### 7.1 Herramientas de IA utilizadas
| Herramienta | Uso | Partes |
|---|---|---|
| ⟨asistente usado, p. ej. Gemini / ChatGPT / Claude⟩ | Organización del plan de trabajo y estructura del notebook | Distribución de tareas, índice |
| ⟨asistente⟩ | Borradores de documentación y del verificador independiente de rutas | Secciones 1, 6 y 7, `README.md` |
| ⟨completar por cada integrante⟩ | ⟨uso⟩ | ⟨sección⟩ |

La IA **no** se usó para producir resultados numéricos: todas las cifras de la sección 6 provienen de la ejecución del notebook.

### 7.2 Cómo se verificó el código generado
- **Lectura y explicación:** cada integrante revisó línea por línea el código de su sección y debe poder explicarlo sin apoyo de la IA.
- **Pruebas con `assert`:** casos pequeños con resultado conocido calculado a mano (costos de transición, vecinos, bloqueos, validaciones).
- **Verificador independiente:** `verificar_ruta` valida cada ruta contra las reglas del problema y recalcula su costo; `existe_camino` (BFS) confirma los casos “sin ruta”. El propio verificador se probó con rutas incorrectas a propósito.
- **Comprobación cruzada:** Dijkstra y A\* deben coincidir en costo óptimo; una diferencia indicaría heurística no admisible o un error de implementación.
- **Revisión visual:** las rutas se inspeccionaron sobre el mapa de calor para confirmar que esquivan bloqueos y zonas de alto riesgo.
- **Reejecución limpia:** `Restart & Run All` antes de cada commit a `main`.

### 7.3 Reproducibilidad
- Python ≥ 3.10 (se usan anotaciones `tuple[int, int]` y `X | Y`).
- Dependencias: `numpy`, `matplotlib`, `jupyter` (ver `requirements.txt`). `heapq`, `math`, `time` y `collections` son de la biblioteca estándar.

```bash
pip install -r requirements.txt
jupyter notebook modelado_terreno.ipynb   # luego Kernel -> Restart & Run All
```